# Standalone ablation evaluation -- CTC-only & AR-only

Evaluates the two standalone ablation checkpoints (`notebooks/train_recognizer_ctc_only.ipynb`'s `ctc_only_standalone` and `notebooks/train_recognizer_ar_only.ipynb`'s `ar_only_standalone`), each trained with no architectural connection to the other -- CTC-only has no AR decoder at all, AR-only has no CTC head at all. Kept in its own notebook, separate from `evaluate_ctc_ar_sequential.ipynb` (which evaluates the combined `Recognizer` checkpoint -- encoder + CTC head + AR decoder together, a structurally different model these two standalone checkpoints can never be loaded as).

Both sections below are fully self-contained -- each defines its own config, its own minimal model class, and its own `common` module shim needed to unpickle that checkpoint's `model_cfg`. Run either section independently after Sections 1/1a/1b (setup) and 2 (data); neither depends on the other having run.

## 1. Setup

In [1]:
import os, subprocess, sys

def detect_environment():
    # Kaggle is checked first: some Kaggle kernels leak a stray COLAB_GPU/
    # COLAB_RELEASE_TAG env var, which would otherwise misdetect as Colab and
    # crash trying to mount Google Drive. "/kaggle/working" existing is a much
    # harder signal to spoof than an env var, so it takes priority.
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working"):
        return "kaggle"
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        return "colab"
    return "local"

ENV = detect_environment()
print("environment:", ENV)

REPO_URL = "https://github.com/Pich09/tuna-ocr.git"
REPO_DIR = "tuna-ocr"

def run_git(args):
    """Runs git and raises with git's ACTUAL stderr on failure. A bare
    CalledProcessError only reports "exit status 128", which is git's catch-all
    and says nothing about which of the many possible causes (existing
    directory, auth, network) actually happened."""
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed ({r.returncode}):\n{r.stderr.strip()}")
    return r

# Three cases, in order. The middle one is the important fix: after a kernel
# restart the cwd resets to /content (or /kaggle/working), so "recognizer" is no
# longer visible even though a previous run already cloned the repo -- the old
# code then tried to clone again and git aborted with "destination path already
# exists and is not an empty directory" (exit 128).
def pull_latest():
    """Fast-forward the clone we're standing in, LOUDLY. A stale clone is the
    single most confusing failure mode of this notebook: the library code is
    older than the notebook cell driving it, so you get a TypeError about an
    unexpected keyword argument for a config field that plainly exists on
    GitHub. Failing to pull is survivable (offline runtime, dirty tree), so
    this doesn't raise -- but it must never be a quiet one-line note."""
    try:
        run_git(["pull", "--ff-only"])
        print("pulled latest changes")
    except RuntimeError as e:
        print("!" * 78)
        print("WARNING: could not update the clone -- running POSSIBLY STALE code.")
        print(f"  {e}")
        print("  If a later cell fails with 'unexpected keyword argument', this is why.")
        print("  Fix: !git -C . fetch origin && git -C . reset --hard origin/main")
        print("       then restart the runtime (stale modules stay imported).")
        print("!" * 78)

if os.path.isdir("recognizer"):
    # Already inside the repo -- which is what re-running this cell in the same
    # session always looks like, since the first run chdir'd here. This branch
    # used to just print and return, so a second run silently kept whatever code
    # the session started with and never saw upstream commits again.
    print(f"already inside the repo working dir: {os.getcwd()}")
    pull_latest()
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"found an existing clone, reusing it: {os.getcwd()}")
    pull_latest()
else:
    run_git(["clone", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    print(f"cloned into {os.getcwd()}")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())
# Print the resolved commit: the one unambiguous answer to "is my library code
# actually the version I think it is?", checkable against the GitHub history.
print("repo commit:  " + run_git(["log", "-1", "--pretty=%h %s"]).stdout.strip())


environment: colab
found an existing clone, reusing it: /content/tuna-ocr
pulled latest changes
working dir: /content/tuna-ocr
repo commit:  39bfda6 evaluate_ctc_ar_sequential: fix Section 3 still pointed at ctc_only_standalone


In [2]:
# Colab and Kaggle both ship torch preinstalled and matched to their runtime (the CUDA
# driver on a GPU runtime, or the torch_xla/libtpu build on a TPU runtime) -- blindly
# `pip install torch` on top of that (e.g. via a plain `-r recognizer/requirements.txt`)
# can silently replace it with a build that doesn't match, which breaks GPU support and
# breaks TPU support even harder (torch_xla is pinned to one exact torch version).
# Install everything else normally, and only pip-install torch if it isn't importable
# at all (a bare local venv).
import importlib.util
from pathlib import Path

def strip_torch(req_path):
    lines = Path(req_path).read_text().splitlines()
    return [l for l in lines if not l.strip().lower().startswith("torch")]

torch_before = None
if importlib.util.find_spec("torch") is not None:
    import torch
    torch_before = torch.__version__

reqs = strip_torch("recognizer/requirements.txt") + strip_torch("real_data/requirements.txt")
Path("/tmp/_notebook_requirements.txt").write_text("\n".join(reqs) + "\n")
!pip install -q -r /tmp/_notebook_requirements.txt

if torch_before is None:
    print("torch not found -- installing (no preinstalled build to preserve here)")
    !pip install -q torch
else:
    # Excluding torch from the requirements file isn't a complete guarantee: any
    # dependency in it is free to pull a *different* torch in as its own dependency.
    # On a TPU runtime that's silently fatal -- torch_xla only loads against the exact
    # torch build it was compiled for, and the failure surfaces much later as an opaque
    # import/libtpu error, so check explicitly here rather than discovering it then.
    # importlib.metadata, not `torch.__version__`: torch is already imported in this
    # kernel, so its module object still reports the OLD version no matter what pip
    # just wrote to disk (and importlib.reload(torch) is not a safe way to find out).
    # The distribution metadata reflects what's actually installed now.
    from importlib.metadata import version as _pkg_version
    torch_after = _pkg_version("torch")
    if torch_after != torch_before:
        print(f"WARNING: pip changed torch {torch_before} -> {torch_after} as a "
              f"transitive dependency. On a TPU runtime, restart the runtime and "
              f"`pip install torch=={torch_before}` before continuing, or torch_xla "
              f"will fail to load.")
    else:
        print(f"using preinstalled torch {torch_after} "
              f"(cuda available: {torch.cuda.is_available()}) -- not reinstalled")


using preinstalled torch 2.11.0+cpu (cuda available: False) -- not reinstalled


In [3]:
import os

from recognizer import env_utils

checkpoint_root = env_utils.get_checkpoint_root(ENV)

# Token resolution, in order. Colab's own secret store (env_utils.get_hf_token) is
# tried first but is NOT reliable: it raises "Secrets can only be fetched when running
# from the Colab UI" whenever the notebook runs detached from the UI tab, which is
# exactly what happened on a long training run here. So fall back to an HF_TOKEN
# environment variable, then to a plain file, then to an interactive prompt --
# deliberately never hardcoded in this notebook, which is committed to a public git
# repo (GitHub's push protection rejects the commit outright, and HF's secret scanner
# auto-revokes any write-scoped token that lands in one).
#
# Easiest on Colab: run this in a scratch cell once per session, paste when prompted:
#     import os, getpass; os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")
hf_token = None
try:
    hf_token = env_utils.get_hf_token(ENV)
except Exception as e:
    print(f"platform secret store unavailable ({type(e).__name__}), trying fallbacks...")
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    for candidate in ("/content/hf_token.txt", "/kaggle/working/hf_token.txt", "hf_token.txt"):
        if os.path.exists(candidate):
            hf_token = open(candidate).read().strip()
            print(f"read HF token from {candidate}")
            break
if not hf_token:
    import getpass
    hf_token = getpass.getpass("HF token (input hidden): ").strip()

# Resolve the accelerator NOW, before the multi-hour cells below, and print the exact
# torch device that training will use. detect_accelerator() covers both TPU
# generations (legacy XRT env vars and current PJRT ones) plus the /dev/accel* device
# nodes, so a modern Colab/Kaggle TPU runtime is recognised rather than falling through
# to CPU -- a fallback that is otherwise invisible until you notice steps taking 100x
# too long, hours in.
accelerator = env_utils.detect_accelerator()
device = env_utils.get_torch_device()

print("environment:      ", ENV)
print("checkpoint root:  ", checkpoint_root)
print("HF token loaded:  ", bool(hf_token))
print("accelerator:      ", env_utils.describe_accelerator())
print("torch device:     ", device)
if accelerator == "cpu":
    print("\n>>> No GPU/TPU detected. Colab: Runtime > Change runtime type. "
          "Kaggle: Settings > Accelerator. Do not start the training cell on CPU -- "
          "at this dataset's scale it will not finish.")


platform secret store unavailable (RuntimeError), trying fallbacks...
environment:       colab
checkpoint root:   /content/drive/My Drive/tuna-ocr/checkpoints
HF token loaded:   True
accelerator:       cpu (no GPU/TPU detected -- training will be impractically slow at this scale)
torch device:      cpu

>>> No GPU/TPU detected. Colab: Runtime > Change runtime type. Kaggle: Settings > Accelerator. Do not start the training cell on CPU -- at this dataset's scale it will not finish.


In [4]:
# Downloads Panhapich/khmer-sp-8k's SentencePiece model + khmer_segmentation.py
# wrapper (a bare .model file is not enough -- see recognizer/README.md).
from recognizer.tokenizer.fetch_tokenizer import fetch_tokenizer

fetch_tokenizer()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetched Panhapich/khmer-sp-8k -> /content/tuna-ocr/recognizer/tokenizer/assets: ['gazetteer.json', 'khmer_segmentation.py', 'khmer_sp.model', 'latin_exceptions.json', 'tokenizer_info.json']


PosixPath('/content/tuna-ocr/recognizer/tokenizer/assets')

## 1a. Patch a tokenizer decode bug (inflates val_ar_cer)

The vendored `khmer_segmentation.py` (downloaded above from `Panhapich/khmer-sp-8k`)
has a `KhmerTokenizer.decode()` that does `self.sp.decode(ids).replace(" ", "")` --
this strips **every** space unconditionally, not just the artificial Khmer
word-boundary spaces `segment_line()` introduces for SentencePiece training. So any
real space in the reference text (English words, mixed Khmer/English text,
punctuation spacing) is deleted from the AR decoder's decoded output regardless of
whether the model predicted it correctly -- inflating `val_ar_cer` for any sample
containing genuine whitespace. `val_ctc_cer` is unaffected (`CharVocab.decode` in
`recognizer/data/char_vocab.py` does no space stripping), so it remains a trustworthy
read on encoder accuracy even without this patch.

This cell rewrites the downloaded `khmer_segmentation.py` on disk (the same file
every `KhmerOcrTokenizer()` construction loads from -- see
`khmer_ocr_tokenizer.py`'s `_load_upstream_khmer_tokenizer`) so `decode()` only
strips a space when it falls strictly between two Khmer characters -- the actual
artificial-boundary case -- and leaves every other space alone. Idempotent: skips if
already patched, so re-running this cell (or a fresh fetch that re-downloads the
original file) is safe.

In [5]:
from recognizer.config import TOKENIZER_ASSETS_DIR

_seg_path = TOKENIZER_ASSETS_DIR / "khmer_segmentation.py"
_seg_src = _seg_path.read_text(encoding="utf-8")

_BUGGY_DECODE = '''    def decode(self, ids) -> str:
        # Strip the artificial word-boundary spaces introduced for training;
        # natural Khmer orthography does not space every word.
        return self.sp.decode(ids).replace(" ", "")'''

_PATCHED_DECODE = '''    def decode(self, ids) -> str:
        # PATCHED (notebook cell 1a): the original body here did
        # `self.sp.decode(ids).replace(" ", "")`, which strips EVERY space --
        # including genuine ones in English words, mixed Khmer/English text, and
        # punctuation spacing -- not just the artificial Khmer word-boundary
        # spaces segment_line() introduces for SentencePiece training. That
        # silently deletes correctly-predicted spaces before CER ever sees them.
        # Only strip a space strictly between two Khmer characters -- the real
        # artificial-boundary case -- and leave every other space alone.
        import re as _re
        decoded = self.sp.decode(ids)
        return _re.sub(r"(?<=[\\u1780-\\u17ff])\\s(?=[\\u1780-\\u17ff])", "", decoded)'''

if _PATCHED_DECODE in _seg_src:
    print(f"{_seg_path} already patched -- nothing to do")
elif _BUGGY_DECODE in _seg_src:
    _seg_path.write_text(_seg_src.replace(_BUGGY_DECODE, _PATCHED_DECODE), encoding="utf-8")
    print(f"patched {_seg_path}: decode() now only strips Khmer-Khmer boundary spaces")
else:
    raise RuntimeError(
        f"{_seg_path} doesn't match the expected buggy decode() body -- the upstream "
        f"file may have changed. Inspect it manually before training: the goal is a "
        f"decode() that doesn't strip every space unconditionally."
    )


/content/tuna-ocr/recognizer/tokenizer/assets/khmer_segmentation.py already patched -- nothing to do


## 1b. Low-memory dataset loading patch

`recognizer/data/manifest.py`'s `load_dedup_arrow` (unmodified library code)
materializes the ENTIRE image-bytes column into a Python list
(`table.column("image").to_pylist()`), then builds a second full list of
`Sample` objects from it -- both lists stay alive simultaneously until the
function returns, so peak memory during dataset loading is roughly **2x**
the dataset's actual image-bytes size. On a Colab session this is enough to
get the kernel OOM-killed mid-load, which surfaces as no Python traceback at
all -- just `"Canceled future for execute_request message before replies
were done"` -- because the process itself dies, not one call inside it.

This cell monkeypatches `load_dedup_arrow` to iterate the Arrow columns
directly instead of pre-snapshotting them, so no intermediate full-column
Python list is ever alive alongside the final result -- same output, roughly
half the peak memory. This is a real fix to shared library code, not a
notebook-only workaround -- worth upstreaming into
`recognizer/data/manifest.py` directly once confirmed, so every consumer of
`run_training` benefits, not just this notebook.

In [6]:
# Monkeypatches recognizer.data.manifest.load_dedup_arrow: safe because
# load_dedup_manifest (which run_training actually calls) looks up
# load_dedup_arrow by name in the module's own namespace at CALL time, not at
# import time -- so reassigning the module attribute here takes effect for
# every call made after this cell runs, without needing to touch train.py or
# re-import anything downstream.
import recognizer.data.manifest as _manifest

def _load_dedup_arrow_low_memory(path):
    import pyarrow as pa

    with pa.memory_map(str(path), "rb") as source:
        table = pa.ipc.open_file(source).read_all()
    text_col = table.column("text")
    source_col = table.column("source")
    image_col = table.column("image")
    # zip() over ChunkedArrays iterates chunk-by-chunk, yielding pa.Scalar
    # objects one at a time -- .as_py() converts just that one value, so at
    # most one row's worth of extra Python objects exists beyond the `samples`
    # list actually being built, vs. the original's three full-column lists
    # PLUS the final list all alive at once.
    samples = []
    for t, s, img in zip(text_col, source_col, image_col):
        samples.append(_manifest.Sample(image_bytes=img.as_py(), text=t.as_py(), source=s.as_py()))
    return samples

_manifest.load_dedup_arrow = _load_dedup_arrow_low_memory
print("patched recognizer.data.manifest.load_dedup_arrow for lower peak memory during dataset loading")

patched recognizer.data.manifest.load_dedup_arrow for lower peak memory during dataset loading


## 2. Data

In [7]:
from pathlib import Path

import pyarrow as pa

from real_data import hf_push
from real_data.config import HF_DATA_REPO_ID, REAL_DATA_ROOT

dedup_raw = REAL_DATA_ROOT / "samples" / "dedup.arrow"
dedup_filtered = REAL_DATA_ROOT / "samples" / "dedup_filtered.arrow"


def _valid_arrow(p: Path) -> bool:
    if not p.exists():
        return False
    try:
        with pa.memory_map(str(p), "rb") as f:
            pa.ipc.open_file(f).schema
        return True
    except pa.ArrowInvalid:
        return False


# --- 2a. get dedup.arrow (download from the Hub if not already local) ---
if _valid_arrow(dedup_raw):
    print(f"{dedup_raw} present and valid -- skipping download")
elif hf_push.dataset_exists_on_hub(HF_DATA_REPO_ID, token=hf_token):
    print(f"downloading prebuilt dataset from {HF_DATA_REPO_ID} ...")
    hf_push.pull_dataset(dedup_raw, token=hf_token, repo_id=HF_DATA_REPO_ID)
    print("downloaded ->", dedup_raw)
else:
    raise RuntimeError(
        f"No local {dedup_raw} and nothing on {HF_DATA_REPO_ID}. Run the DATA "
        f"cells (section 2) of train_recognizer_ctc_ar_sequential.ipynb once to "
        f"build + push the dataset, then re-run this cell."
    )

# --- 2b. re-apply the training notebook's embedded-newline filter (its cell 2c) ---
if _valid_arrow(dedup_filtered):
    print(f"{dedup_filtered} present and valid -- skipping filter pass")
else:
    with pa.memory_map(str(dedup_raw), "rb") as source:
        table = pa.ipc.open_file(source).read_all()
    texts = table.column("text").to_pylist()
    sources = table.column("source").to_pylist()
    keep_mask = [("\n" not in t) for t in texts]
    dropped = {}
    for t, s, keep in zip(texts, sources, keep_mask):
        if not keep:
            dropped[s] = dropped.get(s, 0) + 1
    print(f"newline filter: dropping {len(keep_mask) - sum(keep_mask)}/{len(keep_mask)} "
          f"samples with embedded newlines: {dropped or 'none'}")
    filtered = table.filter(pa.array(keep_mask))
    tmp = dedup_filtered.with_name(dedup_filtered.name + ".tmp")
    with pa.OSFile(str(tmp), "wb") as sink:
        with pa.ipc.new_file(sink, filtered.schema) as writer:
            writer.write_table(filtered)
    tmp.replace(dedup_filtered)
    print(f"wrote {dedup_filtered}")

dedup_manifest = dedup_filtered
print("evaluation will use:", dedup_manifest)


/content/tuna-ocr/real_data/samples/dedup.arrow present and valid -- skipping download
/content/tuna-ocr/real_data/samples/dedup_filtered.arrow present and valid -- skipping filter pass
evaluation will use: /content/tuna-ocr/real_data/samples/dedup_filtered.arrow


## 3. CTC-only ablation checkpoint

`ctc_only_standalone` / `v2_ctc_only_standalone` -- encoder + CTC head only. Uses the character-level `CharTokenizer` bundled inside the checkpoint itself (see `notebooks/train_recognizer_ctc_only.ipynb`'s `ctc_train.py` -- this tokenizer's vocab is rebuilt from training data each session, so it's saved alongside the weights rather than assumed fixed).

In [ ]:
# ---- CTC-only ablation config ----
CTCONLY_REPO_ID  = "Panhapich/Tuna-OCR"        # renamed from Panhapich/tuna-ocr -- see git history
CTCONLY_PREFIX   = "ctc_only_standalone"
CTCONLY_RUN_NAME = "v2_ctc_only_standalone"
CTCONLY_WHICH    = "best"                       # "best" or "latest"
CTCONLY_BATCH_SIZE = 16
CTCONLY_SHOW_WORST = 5
# ----------------------------------

# Fully self-contained -- does NOT depend on section 3/3b/4 having been run (those
# load the DIFFERENT ctc_ar_sequential checkpoint via recognizer.evaluate.load_model,
# a full Recognizer with both heads; this section evaluates the CTC-only standalone
# checkpoint, a structurally different, smaller model). Only needs section 1/1a/1b
# (setup) and section 2 (data) to have run first.
import csv
import random
import sys
import time
import types
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import editdistance
import torch
import torch.nn as nn

from recognizer.config import ModelConfig as _RecModelConfig, TrainConfig as _RecTrainConfig
from recognizer.data.dataset import find_unlearnable
from recognizer.data.manifest import load_dedup_manifest
from recognizer.data.transforms import chunk_line_image
from recognizer.hf_push import pull_best_checkpoint, pull_latest_checkpoint
from recognizer.modules.encoder import ConformerEncoder as _RecConformerEncoder
from recognizer.tokenizer.char_tokenizer import CharTokenizer as _CharTokenizer

_co_tc = _RecTrainConfig()
CTCONLY_SEED, CTCONLY_VAL_FRAC = _co_tc.seed, _co_tc.val_frac  # the values run_training actually used


def _co_cer(pairs):
    return sum(editdistance.eval(r, h) for r, h in pairs) / (sum(len(r) for r, _ in pairs) or 1)


def _co_exact(pairs):
    return sum(r == h for r, h in pairs) / (len(pairs) or 1)


def collapse_ctc(row, blank_id):
    out, prev = [], None
    for c in row:
        if c != prev and c != blank_id:
            out.append(c)
        prev = c
    return out


# The standalone checkpoints pickle their OWN `common.ModelConfig` dataclass; shim
# that module so torch.load can unpickle state["model_cfg"] (we only read fields off
# it, then rebuild a recognizer.config.ModelConfig from those values).
if "common" not in sys.modules:
    _shim = types.ModuleType("common")

    @dataclass
    class ModelConfig:  # field names must match the standalone's common.ModelConfig
        img_height: int = 64
        chunk_width: int = 128
        chunk_overlap: int = 16
        d_model: int = 256
        num_encoder_layers: int = 8
        encoder_attn_heads: int = 4
        encoder_conv_kernel: int = 15
        encoder_ff_expansion: int = 4
        encoder_dropout: float = 0.1
        num_decoder_layers: int = 4
        decoder_attn_heads: int = 4

    class TrainConfig:  # not stored as an instance, but shim it anyway
        pass

    _shim.ModelConfig = ModelConfig
    _shim.TrainConfig = TrainConfig
    sys.modules["common"] = _shim

_co_dir = Path(checkpoint_root) / CTCONLY_RUN_NAME
_co_dir.mkdir(parents=True, exist_ok=True)
_co_pull = pull_best_checkpoint if CTCONLY_WHICH == "best" else pull_latest_checkpoint
_co_path = _co_pull(_co_dir, token=hf_token, repo_id=CTCONLY_REPO_ID, path_prefix=CTCONLY_PREFIX)
assert _co_path is not None, f"no {CTCONLY_WHICH}.pt under {CTCONLY_REPO_ID}/{CTCONLY_PREFIX}"
print("ctc-only checkpoint:", _co_path)

_co_state = torch.load(_co_path, map_location="cpu", weights_only=False)
_mc = _co_state["model_cfg"]
_co_cfg = _RecModelConfig(
    img_height=_mc.img_height, chunk_width=_mc.chunk_width, chunk_overlap=_mc.chunk_overlap,
    d_model=_mc.d_model, num_encoder_layers=_mc.num_encoder_layers,
    encoder_attn_heads=_mc.encoder_attn_heads, encoder_conv_kernel=_mc.encoder_conv_kernel,
    encoder_ff_expansion=_mc.encoder_ff_expansion, encoder_dropout=_mc.encoder_dropout)
ctconly_tok = _CharTokenizer.from_json(_co_state["char_tokenizer"])
print(f"ctc-only vocab: {ctconly_tok.size} classes (incl. blank id {ctconly_tok.blank_id})")


def _co_scaled_width(s):
    from recognizer.data.transforms import open_image
    with open_image(s.image_source) as im:
        w, h = im.size
    return max(1, round(w * _co_cfg.img_height / max(1, h)))


class _CTCOnlyModel(nn.Module):
    def __init__(self, cfg, ctc_vocab_size):
        super().__init__()
        self.encoder = _RecConformerEncoder(cfg)
        self.ctc_head = nn.Linear(cfg.d_model, ctc_vocab_size)

    def encode(self, chunk_batch, chunks_per_line):
        return self.encoder(chunk_batch, chunks_per_line)


ctconly_model = _CTCOnlyModel(_co_cfg, ctconly_tok.size).to(device)
_missing, _unexpected = ctconly_model.load_state_dict(_co_state["model_state_dict"], strict=False)
print("load_state_dict  missing:", list(_missing), " unexpected:", list(_unexpected))
ctconly_model.eval()


@torch.no_grad()
def _ctconly_decode(group):
    ordered = sorted(group, key=_co_scaled_width)
    rows = []
    for i in range(0, len(ordered), CTCONLY_BATCH_SIZE):
        batch = ordered[i:i + CTCONLY_BATCH_SIZE]
        chunks, cpl = [], []
        for s in batch:
            ct, _ = chunk_line_image(s.image_source, _co_cfg.chunk_width,
                                     _co_cfg.chunk_overlap, _co_cfg.img_height)
            chunks.extend(ct)
            cpl.append(len(ct))
        enc_out, enc_lengths, _ = ctconly_model.encode(
            torch.stack(chunks).to(device), torch.tensor(cpl, dtype=torch.long))
        ids = ctconly_model.ctc_head(enc_out).argmax(-1).cpu().tolist()
        lens = enc_lengths.cpu().tolist()
        for s, row, n in zip(batch, ids, lens):
            rows.append((s.text, ctconly_tok.decode(collapse_ctc(row[:n], ctconly_tok.blank_id))))
    return rows


# same held-out split logic section 3b uses, filtered with THIS run's own char vocab
_co_samples = list(load_dedup_manifest(dedup_manifest))
random.Random(CTCONLY_SEED).shuffle(_co_samples)
_co_val = _co_samples[:max(1, int(len(_co_samples) * CTCONLY_VAL_FRAC))]
_co_bad = set(find_unlearnable(_co_val, _co_cfg, ctconly_tok))
_co_val = [s for i, s in enumerate(_co_val) if i not in _co_bad]
print(f"{len(_co_samples)} total -> {len(_co_val)} val (dropped {len(_co_bad)} unlearnable)")

_co_groups = defaultdict(list)
for s in _co_val:
    _co_groups[s.source].append(s)

print(f"\n{'source':40}{'n':>8}{'CTC CER':>10}{'CTC exact':>12}")
print("-" * 70)
_co_all, _co_rows = [], []
for src in sorted(_co_groups):
    t0 = time.time()
    dec = _ctconly_decode(_co_groups[src])
    _co_all += dec
    print(f"{src[:40]:40}{len(dec):8d}{_co_cer(dec):10.4f}{_co_exact(dec):12.1%}   [{time.time() - t0:.0f}s]")
    _co_rows.append([f"ctconly:{src}", "", "", "", len(dec), _co_cer(dec), _co_exact(dec)])
    for ref, hyp in sorted(dec, key=lambda p: editdistance.eval(p[0], p[1]), reverse=True)[:CTCONLY_SHOW_WORST]:
        print(f"    gt : {ref!r}")
        print(f"    ctc: {hyp!r}")
print("-" * 70)
print(f"{'ALL (micro-avg)':40}{len(_co_all):8d}{_co_cer(_co_all):10.4f}{_co_exact(_co_all):12.1%}")
_co_rows.append(["ctconly:ALL", "", "", "", len(_co_all), _co_cer(_co_all), _co_exact(_co_all)])

# Its own CSV, independent of section 3's -- avoids depending on _csv_rows/flush_csv
# (only defined if section 3 ran) or clobbering section 3's own eval_results.csv.
CTCONLY_CSV_PATH = Path(checkpoint_root) / CTCONLY_RUN_NAME / "eval_results.csv"
with open(CTCONLY_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["group", "n_ar", "ar_cer", "ar_exact", "n_ctc", "ctc_cer", "ctc_exact"])
    w.writerows(_co_rows)
print("\nsaved", CTCONLY_CSV_PATH)


## 4. AR-only ablation checkpoint

`ar_only_standalone` / `v2_ar_only_standalone` -- encoder + sequential AR decoder only, no CTC head anywhere. Trained with teacher forcing (see `notebooks/train_recognizer_ar_only.ipynb`'s `ar_decoder.py` docstring for why); this eval, like real inference, decodes autoregressively on the model's own greedily-generated tokens (`decode_greedy`) -- teacher forcing only ever shaped training, never evaluation. Uses the shared, fixed SentencePiece tokenizer (`Panhapich/khmer-sp-8k`, fetched in section 1a above), not `CharTokenizer`.

In [ ]:
# ---- AR-only ablation config ----
ARONLY_REPO_ID  = "Panhapich/Tuna-OCR"
ARONLY_PREFIX   = "ar_only_standalone"
ARONLY_RUN_NAME = "v2_ar_only_standalone"
ARONLY_WHICH    = "best"                       # "best" or "latest"
ARONLY_BATCH_SIZE = 16
ARONLY_MAX_DECODE_LEN = 256                     # matches ARDecoder.decode_greedy's default
ARONLY_SHOW_WORST = 5
# ----------------------------------

# Fully self-contained, like the CTC-only section above -- does NOT depend on
# section 3/3b/4 (the DIFFERENT ctc_ar_sequential checkpoint, a full
# Recognizer with both heads) or on the CTC-only section having run. This
# section evaluates the AR-only standalone checkpoint: encoder + a sequential
# AR decoder ONLY, no CTC head anywhere. Trained with teacher forcing (see
# notebooks/train_recognizer_ar_only.ipynb's ar_decoder.py docstring); this
# eval, like real inference, decodes autoregressively on the model's own
# tokens (decode_greedy) -- teacher forcing only ever shaped training.
# Tokenizer: the shared SentencePiece tokenizer (Panhapich/khmer-sp-8k, fixed/
# pretrained -- fetched in section 1a above), NOT CharTokenizer -- unlike the
# CTC-only checkpoint, nothing is bundled in the checkpoint for this, and
# nothing needs to be: fetch_tokenizer() always yields the identical vocab.
import sys
import time
import types
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import editdistance
import torch
import torch.nn as nn

from recognizer.config import ModelConfig as _RecModelConfig, TOKENIZER_ASSETS_DIR, TrainConfig as _RecTrainConfig
from recognizer.data.manifest import load_dedup_manifest
from recognizer.data.transforms import chunk_line_image, open_image
from recognizer.hf_push import pull_best_checkpoint, pull_latest_checkpoint
from recognizer.modules.attention import MultiHeadAttention
from recognizer.modules.conformer_block import FeedForward
from recognizer.modules.decoder import truncate_at_eos
from recognizer.modules.encoder import ConformerEncoder as _RecConformerEncoder
from recognizer.modules.positional import SinusoidalPositionalEncoding
from recognizer.tokenizer.khmer_ocr_tokenizer import KhmerOcrTokenizer

_ar_tc = _RecTrainConfig()
ARONLY_SEED, ARONLY_VAL_FRAC = _ar_tc.seed, _ar_tc.val_frac  # values run_training actually used


def _ar_cer(pairs):
    return sum(editdistance.eval(r, h) for r, h in pairs) / (sum(len(r) for r, _ in pairs) or 1)


def _ar_exact(pairs):
    return sum(r == h for r, h in pairs) / (len(pairs) or 1)


def _ar_scaled_width(s):
    with open_image(s.image_source) as im:
        w, h = im.size
    return max(1, round(w * _ar_cfg.img_height / max(1, h)))


# The standalone checkpoints pickle their OWN `common.ModelConfig` dataclass; shim
# that module so torch.load can unpickle state["model_cfg"] (only field values are
# read off it, then a recognizer.config.ModelConfig is rebuilt from those). Same
# shim as the CTC-only section above -- harmless to redefine if it already ran.
if "common" not in sys.modules:
    _shim = types.ModuleType("common")

    @dataclass
    class ModelConfig:  # field names must match the standalone's common.ModelConfig
        img_height: int = 64
        chunk_width: int = 128
        chunk_overlap: int = 16
        d_model: int = 256
        num_encoder_layers: int = 8
        encoder_attn_heads: int = 4
        encoder_conv_kernel: int = 15
        encoder_ff_expansion: int = 4
        encoder_dropout: float = 0.1
        num_decoder_layers: int = 4
        decoder_attn_heads: int = 4

    class TrainConfig:  # not stored as an instance, but shim it anyway
        pass

    _shim.ModelConfig = ModelConfig
    _shim.TrainConfig = TrainConfig
    sys.modules["common"] = _shim

_ar_dir = Path(checkpoint_root) / ARONLY_RUN_NAME
_ar_dir.mkdir(parents=True, exist_ok=True)
_ar_pull = pull_best_checkpoint if ARONLY_WHICH == "best" else pull_latest_checkpoint
_ar_path = _ar_pull(_ar_dir, token=hf_token, repo_id=ARONLY_REPO_ID, path_prefix=ARONLY_PREFIX)
assert _ar_path is not None, f"no {ARONLY_WHICH}.pt under {ARONLY_REPO_ID}/{ARONLY_PREFIX}"
print("ar-only checkpoint:", _ar_path)

_ar_state = torch.load(_ar_path, map_location="cpu", weights_only=False)
_amc = _ar_state["model_cfg"]
_ar_cfg = _RecModelConfig(
    img_height=_amc.img_height, chunk_width=_amc.chunk_width, chunk_overlap=_amc.chunk_overlap,
    d_model=_amc.d_model, num_encoder_layers=_amc.num_encoder_layers,
    encoder_attn_heads=_amc.encoder_attn_heads, encoder_conv_kernel=_amc.encoder_conv_kernel,
    encoder_ff_expansion=_amc.encoder_ff_expansion, encoder_dropout=_amc.encoder_dropout)
_ar_num_decoder_layers = _amc.num_decoder_layers
_ar_decoder_attn_heads = _amc.decoder_attn_heads

artok = KhmerOcrTokenizer(TOKENIZER_ASSETS_DIR)
print(f"ar-only vocab: {artok.vocab_size} pieces "
      f"(pad={artok.pad_id} bos={artok.bos_id} eos={artok.eos_id})")


class _AROnlyDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ff_expansion, dropout):
        super().__init__()
        self.ln_self = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln_cross = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, ff_expansion, dropout)

    def forward(self, x, enc_out, self_mask, cross_mask):
        h = self.ln_self(x)
        x = x + self.self_attn(h, h, h, attn_mask=self_mask)
        h = self.ln_cross(x)
        x = x + self.cross_attn(h, enc_out, enc_out, attn_mask=cross_mask)
        x = x + self.ff(x)
        return x


class _AROnlyDecoder(nn.Module):
    """Inference-only port of ar_decoder.py's ARDecoder -- decode_greedy plus
    the _step it depends on. forward_teacher_forced is dropped: this notebook
    never trains, only evaluates."""

    def __init__(self, cfg, vocab_size, bos_id, pad_id, eos_id, num_decoder_layers, decoder_attn_heads):
        super().__init__()
        self.bos_id, self.pad_id, self.eos_id = bos_id, pad_id, eos_id
        self.token_emb = nn.Embedding(vocab_size, cfg.d_model, padding_idx=pad_id)
        self.pos_enc = SinusoidalPositionalEncoding(cfg.d_model)
        self.layers = nn.ModuleList([
            _AROnlyDecoderLayer(cfg.d_model, decoder_attn_heads, cfg.encoder_ff_expansion, cfg.encoder_dropout)
            for _ in range(num_decoder_layers)
        ])
        self.ln_final = nn.LayerNorm(cfg.d_model)
        self.token_head = nn.Linear(cfg.d_model, vocab_size)

    def _step(self, generated, enc_out, cross_mask):
        device = generated.device
        l_dec = generated.shape[1]
        x = self.token_emb(generated) + self.pos_enc(l_dec).unsqueeze(0)
        causal = torch.tril(torch.ones(l_dec, l_dec, dtype=torch.bool, device=device)).unsqueeze(0).unsqueeze(0)
        for layer in self.layers:
            x = layer(x, enc_out, causal, cross_mask)
        x = self.ln_final(x)
        return self.token_head(x[:, -1, :])

    @torch.no_grad()
    def decode_greedy(self, enc_out, enc_lengths, max_len=256):
        b = enc_out.shape[0]
        device = enc_out.device
        t_enc = enc_out.shape[1]
        enc_len_mask = torch.arange(t_enc, device=device).unsqueeze(0) < enc_lengths.unsqueeze(1)
        cross_mask = enc_len_mask.unsqueeze(1).unsqueeze(1)
        generated = torch.full((b, 1), self.bos_id, dtype=torch.long, device=device)
        finished = torch.zeros(b, dtype=torch.bool, device=device)
        for _ in range(max_len):
            logits = self._step(generated, enc_out, cross_mask)
            next_tok = logits.argmax(dim=-1, keepdim=True)
            next_tok = next_tok.masked_fill(finished.unsqueeze(1), self.pad_id)
            finished = finished | (next_tok.squeeze(1) == self.eos_id)
            generated = torch.cat([generated, next_tok], dim=1)
            if bool(finished.all()):
                break
        return generated[:, 1:]


class _AROnlyModel(nn.Module):
    def __init__(self, cfg, vocab_size, bos_id, pad_id, eos_id, num_decoder_layers, decoder_attn_heads):
        super().__init__()
        self.encoder = _RecConformerEncoder(cfg)
        self.decoder = _AROnlyDecoder(cfg, vocab_size, bos_id, pad_id, eos_id,
                                      num_decoder_layers, decoder_attn_heads)

    def encode(self, chunk_batch, chunks_per_line):
        return self.encoder(chunk_batch, chunks_per_line)


aronly_model = _AROnlyModel(_ar_cfg, artok.vocab_size, artok.bos_id, artok.pad_id, artok.eos_id,
                            _ar_num_decoder_layers, _ar_decoder_attn_heads).to(device)
_missing, _unexpected = aronly_model.load_state_dict(_ar_state["model_state_dict"], strict=False)
print("load_state_dict  missing:", list(_missing), " unexpected:", list(_unexpected))
aronly_model.eval()


@torch.no_grad()
def _aronly_decode(group):
    ordered = sorted(group, key=_ar_scaled_width)
    rows = []
    for i in range(0, len(ordered), ARONLY_BATCH_SIZE):
        batch = ordered[i:i + ARONLY_BATCH_SIZE]
        chunks, cpl = [], []
        for s in batch:
            ct, _ = chunk_line_image(s.image_source, _ar_cfg.chunk_width,
                                     _ar_cfg.chunk_overlap, _ar_cfg.img_height)
            chunks.extend(ct)
            cpl.append(len(ct))
        enc_out, enc_lengths, _ = aronly_model.encode(
            torch.stack(chunks).to(device), torch.tensor(cpl, dtype=torch.long))
        gen = aronly_model.decoder.decode_greedy(enc_out, enc_lengths, max_len=ARONLY_MAX_DECODE_LEN).tolist()
        for s, row in zip(batch, gen):
            trimmed = truncate_at_eos(row, artok.eos_id, artok.pad_id)
            rows.append((s.text, artok.decode(trimmed, strip_control=True)))
    return rows


# Same train/val split ar_train.py's run_training uses: plain seeded shuffle,
# no find_unlearnable filtering (that filter is CTC-target-length specific --
# see recognizer/data/dataset.py's docstring -- meaningless for a model with
# no CTC head at all).
_ar_samples = list(load_dedup_manifest(dedup_manifest))
import random as _ar_random
_ar_random.Random(ARONLY_SEED).shuffle(_ar_samples)
_ar_val = _ar_samples[:max(1, int(len(_ar_samples) * ARONLY_VAL_FRAC))]
print(f"{len(_ar_samples)} total -> {len(_ar_val)} val")

_ar_groups = defaultdict(list)
for s in _ar_val:
    _ar_groups[s.source].append(s)

print(f"\n{'source':40}{'n':>8}{'AR CER':>10}{'AR exact':>12}")
print("-" * 70)
_ar_all, _ar_rows = [], []
for src in sorted(_ar_groups):
    t0 = time.time()
    dec = _aronly_decode(_ar_groups[src])
    _ar_all += dec
    print(f"{src[:40]:40}{len(dec):8d}{_ar_cer(dec):10.4f}{_ar_exact(dec):12.1%}   [{time.time() - t0:.0f}s]")
    _ar_rows.append([f"aronly:{src}", len(dec), _ar_cer(dec), _ar_exact(dec)])
    for ref, hyp in sorted(dec, key=lambda p: editdistance.eval(p[0], p[1]), reverse=True)[:ARONLY_SHOW_WORST]:
        print(f"    gt: {ref!r}")
        print(f"    ar: {hyp!r}")
print("-" * 70)
print(f"{'ALL (micro-avg)':40}{len(_ar_all):8d}{_ar_cer(_ar_all):10.4f}{_ar_exact(_ar_all):12.1%}")
_ar_rows.append(["aronly:ALL", len(_ar_all), _ar_cer(_ar_all), _ar_exact(_ar_all)])

import csv
ARONLY_CSV_PATH = Path(checkpoint_root) / ARONLY_RUN_NAME / "eval_results.csv"
with open(ARONLY_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["group", "n", "ar_cer", "ar_exact"])
    w.writerows(_ar_rows)
print("\nsaved", ARONLY_CSV_PATH)
